# **Osservatorio Raccolta Rifiuti**:

Installazione delle librerie di progetto

In [ ]:
!pip install pandas sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.3 MB/s eta 0:00:00


Eseguo il mount della cartella di Google Drive ove sono presenti i dataset da caricare ed imposto il flag per indicare se il salvataggio del dataset elaborato deve essere effettuato su database oppure su file excel

In [ ]:
from google.colab import drive
# Smonta il drive per azzerare la cache di sessione
drive.flush_and_unmount()
# Eseguo il mount del drive Google
drive.mount('/content/drive')
# Flag per indicare se il salvataggio del dataset di produzione deve essere
# eseguito su database oppure su file excel
FLAG_SALVATAGGIO_DB = False

Mounted at /content/drive


Creo la connessione verso il database postgres ospitato su servizio cloud (SaaS) di *supabase.com*

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy import text
from google.colab import userdata

# Configurazione database
DB_USER = "postgres.luhxmgsxvbkfuylgkthr"
# Inserire nei secret di colab la password per l'accesso al database
DB_PASSWORD = userdata.get("SUPABASE_PASSWORD")
DB_HOST = "aws-0-eu-west-1.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"

# Connessione a Postgres tramite SQLAlchemy
DATABASE_URL = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Creo l'engine tramite la funzione create_engine di SQLAlchemy
engine = create_engine(DATABASE_URL)

Carico tutti i file presenti nella cartella Google Drive in un dataframe pandas

In [ ]:
import os as os
import pandas as pd

# Nome della tabella di produzione su database
TAB_RACCOLTA_DIFFERENZIATA_PROD = "raccolta_differenziata"
TAB_IMBALLAGGI_PROD = "imballaggi"

# Configurazione percorso su google drive ove sono presenti i dataset
CARTELLA_OSSERVATORIO = "/content/drive/MyDrive/Ricicla(MI)/osservatorio_raccolta_differenziata/dataset/"
# Configurazione della cartella su google drive dove salvare i dataset di produzione
CARTELLA_OUTPUT = "/content/drive/MyDrive/Ricicla(MI)/output/"

# Dataset
FILE_RACCOLTA_URBANA = "raccolta_urbana_totale_2020_2023.csv"
FILE_IMBALLAGGI = "imballaggi_totale_2020_2023.csv"

# File excel di output
FILE_RACCOLTA_DIFFERENZIATA_OUT = "raccolta_differenziata.xlsx"
FILE_IMBALLAGGI_OUT = "imballaggi.xlsx"

# Verifica che la cartella dove sono presenti i dataset esista
if not os.path.exists(CARTELLA_OSSERVATORIO):
    print(
        f"ERRORE: La cartella '{CARTELLA_OSSERVATORIO}' non esiste. Assicurati di avere i permessi nececessari per l'accesso a Google Drive."
    )
else:
  # Carico i dataset in un dataframe pandas
  print(f"-> Leggo il file '{FILE_RACCOLTA_URBANA}'")
  df_raccolta = pd.read_csv(CARTELLA_OSSERVATORIO + FILE_RACCOLTA_URBANA, sep=";")
  print(f"-> Leggo il file '{FILE_IMBALLAGGI}'")
  df_imballaggi = pd.read_csv(CARTELLA_OSSERVATORIO + FILE_IMBALLAGGI, sep=";")


-> Leggo il file 'raccolta_urbana_totale_2020_2023.csv'


/tmp/ipykernel_1083/311714449.py:30: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raccolta = pd.read_csv(CARTELLA_OSSERVATORIO + FILE_RACCOLTA_URBANA, sep=";")


-> Leggo il file 'imballaggi_totale_2020_2023.csv'


Modifico/correggo i dati nel Dataframe di Pandas e li salvo nella tabella di produzione su database oppure in un file excel

In [ ]:
# Correggo i dati presenti nel dataset
try:
  df_raccolta["Codice ISTAT"] = df_raccolta["Codice ISTAT"].astype(str).str.zfill(6)
  df_raccolta['Abitanti'] = pd.to_numeric(df_raccolta['Abitanti'], errors='coerce')
  df_raccolta['Codice CER'] = df_raccolta['Codice CER'].astype(str).str.replace('-', '', regex=False).str.strip()
  df_raccolta.loc[df_raccolta['Codice CER'] != 'nan', 'Codice CER'] = df_raccolta.loc[df_raccolta['Codice CER'] != 'nan', 'Codice CER'].str.ljust(6, '0')

  df_raccolta['Raccolta (Kg)'] = df_raccolta['Raccolta (Kg)'].str.replace('.', '', regex=False).str.strip()
  df_raccolta['Raccolta (Kg)'] = pd.to_numeric(df_raccolta['Raccolta (Kg)'], errors='coerce')
  df_raccolta['Anno'] = pd.to_numeric(df_raccolta['Anno'], errors='coerce')
except Exception as e:
  print(f"Eccezione: {e}")

# Rinomino le colonne
df_raccolta.columns = ['codice_istat', 'denominazione', 'abitanti', 'codice_cer', 'raccolta', 'anno', 'regione']
print(f"Colonne del dataframe rinominate")

Colonne del dataframe rinominate


Eseguo controlli di data quality:


*   verifico che la combinazione 'codice_istat', 'anno' e 'codice_cer' non duplicata
*   verifico che non vi siano valori nulli nelle colonne del dataframe
* verifico che la colonna 'anno' appartenente al dominio chiuso [2020, 2021, 2022, 2023]



In [ ]:
# Controllo: Combinazione 'codice_istat', 'anno' e 'codice_cer' non duplicata
# Restituisce le righe che sono duplicate rispetto a queste tre colonne
duplicati = df_raccolta[df_raccolta.duplicated(subset=['codice_istat', 'anno','codice_cer'], keep=False)]
if not duplicati.empty:
    print(f"[FAIL] Trovate {len(duplicati)} righe duplicate per 'codice_istat', 'anno', 'codice_cer':")
    print(duplicati[['codice_istat', 'anno', 'codice_cer']], "\n")
else:
    print("[PASS] Nessun duplicato trovato per 'codice_istat', 'anno', 'codice_cer'.")


# Controllo se ci sono valori nulli: restituisce True se ALMENO una colonna ha valori nulli
ha_valori_nulli = df_raccolta.isna().any().any()

if ha_valori_nulli:
    print("[FAIL] Ci sono valori nulli nel DataFrame.\n")

    # Conteggio dei nulli per ogni colonna (mostra solo quelle con errori)
    conteggio_nulli = df_raccolta.isna().sum()
    colonne_con_nulli = conteggio_nulli[conteggio_nulli > 0]

    print("Colonne con valori nulli e relativo conteggio:")
    print(colonne_con_nulli)

    # Il comune con codice istat 1168 ha denominazione nulla
    # correggo manualmente il campo denominaizone nel dataframe
    df_raccolta.loc[df_raccolta['codice_istat'] == "001168", 'denominazione'] = 'Torino'
    print("[PASS] La colonna denominazione con codice istat 001168 è stata corretta.")
else:
    print("[PASS] Ottimo! Tutte le colonne del DataFrame sono NOT NULL.")


# Controllo: 'anno' appartenente al dominio chiuso [2020, 2021, 2022, 2023]
dominio_anni = {2020, 2021, 2022, 2023}
# Controlla se i valori sono presenti nel set del dominio
fuori_dominio = df_raccolta[~df_raccolta['anno'].isin(dominio_anni)]
if not fuori_dominio.empty:
    print(f"[FAIL] Trovati {len(fuori_dominio)} valori in 'anno' fuori dal dominio consentito:")
    print(fuori_dominio[['anno']], "\n")
else:
    print("[PASS] Tutti i valori in 'anno' appartengono al dominio consentito.")


[PASS] Nessun duplicato trovato per 'codice_istat', 'anno', 'codice_cer'.
[FAIL] Ci sono valori nulli nel DataFrame.

Colonne con valori nulli e relativo conteggio:
denominazione    88
dtype: int64
[PASS] La colonna denominazione con codice istat 001168 è stata corretta.
[PASS] Tutti i valori in 'anno' appartengono al dominio consentito.


Carico il Dataframe Pandas elaborato nella tabella di produzione

In [ ]:
# Questa sezione elimina i valori duplicati di codice_istat/anno/codice_cer che
# sono chiave primaria nella tabella di produzione Conta quanti record sono
# considerati duplicati
numero_duplicati = df_raccolta.duplicated(subset=['codice_istat', 'anno', 'codice_cer']).sum()
print(f"Nel DataFrame ci sono {numero_duplicati} record duplicati basati su codice, anno, codice_cer.")
# Estrae tutte le righe che hanno combinazioni duplicate di codice, anno e codice_cer
df_duplicati = df_raccolta[df_raccolta.duplicated(subset=['codice_istat', 'anno', 'codice_cer'], keep=False)]
# Ordina il risultato per rendere visivamente immediato il confronto tra i duplicati
df_duplicati_ordinato = df_duplicati.sort_values(by=['codice_istat', 'anno', 'codice_cer'])
# Mostra i record duplicati
print("Ecco l'elenco dei record duplicati:")
print(df_duplicati_ordinato)

# Elimino i record duplicati
df_raccolta = df_raccolta.drop_duplicates(subset=['codice_istat', 'anno', 'codice_cer'])

if FLAG_SALVATAGGIO_DB:
# Creo la nuova tabella di produzione sovrascrivendo eventualmente quella già
# esistente
  df_raccolta.to_sql(
      name= TAB_RACCOLTA_DIFFERENZIATA_PROD,
      con=engine,
      if_exists='replace',
      index=False,
      chunksize=10000,     # Suddivide l'inserimento in blocchi da 10.000 righe
      method='multi'       # Esegue INSERT multiple accumulate, riducendo i tempi di rete
  )
  print(f"DataFrame salvato su database")
else:
  df_raccolta.to_excel(CARTELLA_OUTPUT + FILE_RACCOLTA_DIFFERENZIATA_OUT, index=False)
  print(f"DataFrame salvato nel file Excel {FILE_RACCOLTA_DIFFERENZIATA_OUT}")
# Metto a null il dataframe
df_raccolta[:] = np.nan


Nel DataFrame ci sono 0 record duplicati basati su codice, anno, codice_cer.
Ecco l'elenco dei record duplicati:
Empty DataFrame
Columns: [codice_istat, denominazione, abitanti, codice_cer, raccolta, anno, regione]
Index: []
DataFrame salvato nel file Excel raccolta_differenziata.xlsx


Carico i dati relativi agli IMBALLAGGI

In [ ]:
import numpy as np

# Sistemo i datatype delle colonne del dataframe
try:
  df_imballaggi["Codice ISTAT"] = df_imballaggi["Codice ISTAT"].astype(str).str.zfill(6)
  df_imballaggi['Denominazione comune'] = df_imballaggi['Denominazione comune'].astype(str)
  df_imballaggi['Abitanti'] = df_imballaggi['Abitanti'].str.replace('.', '', regex=False).astype(int)
  df_imballaggi['Imballaggi kg procapite'] = df_imballaggi['Imballaggi kg procapite'].str.replace('.', '', regex=False)
  df_imballaggi['Imballaggi kg procapite'] = df_imballaggi['Imballaggi kg procapite'].str.replace(',', '.', regex=False).astype(float)
  df_imballaggi['Corrispettivo €/ab.'] = df_imballaggi['Corrispettivo €/ab.'].str.replace(',', '.', regex=False).astype(float)
  df_imballaggi['Regione'] = df_imballaggi['Regione'].astype(str)
  df_imballaggi['Materiale'] = df_imballaggi['Materiale'].astype(str)
  df_imballaggi['Anno'] = pd.to_numeric(df_imballaggi['Anno'], errors='coerce').astype('Int64')
except Exception as e:
  print(f"Eccezione: {e}")

# Rinomino le colonne
df_imballaggi.columns = ['codice_istat', 'denominazione', 'abitanti', 'imballaggi_procapite', 'corrispettivo', 'regione', 'materiale', 'anno', ]
print(f"Colonne del dataframe rinominate")

# data quality -todo

if FLAG_SALVATAGGIO_DB:
# Salvo il dataframe modificato nella tabella di produzione
  df_imballaggi.to_sql(
      name= TAB_IMBALLAGGI_PROD,
      con=engine,
      if_exists='replace',
      index=False,
      chunksize=10000,
      method='multi'
  )
  print(f"DataFrame salvato su database")
else:
  df_imballaggi.to_excel(CARTELLA_OUTPUT + FILE_IMBALLAGGI_OUT, index=False)
  print(f"DataFrame salvato nel file Excel {FILE_IMBALLAGGI_OUT}")
# Metto a null il dataframe
df_imballaggi[:] = np.nan


Colonne del dataframe rinominate
DataFrame salvato nel file Excel imballaggi.xlsx


Chiudo la connessione e rilascio le risorse del pool

In [ ]:
# Chiude la connessione e rilascia le risorse del pool
engine.dispose()